# RSNA Knee Research Candidate — Tony/Sakhawat V1 (public LB 0.836)

SPDX-License-Identifier: Apache-2.0

This inference notebook is derived from two public Kaggle notebooks released under the Apache License 2.0:

1. Tony Li, [RSNA Knee infer, Version 1](https://www.kaggle.com/code/tonylica/rsna-knee-infer?scriptVersionId=340809629), the checkpoint-compatible inference and model architecture. Pinned code SHA-256: `9c460f1f1387f2c7e9bfcf26d24fb10663df80f8cdab4e738c50cb398b46fad7`.
2. Sakhawat Hossen, [Knee RSNA, Version 1](https://www.kaggle.com/code/sakhawathossen/knee-rsna?scriptVersionId=340853917), the overlapping-window TTA and hybrid rank ensemble. Pinned notebook SHA-256: `fc5465a8b7bc1e36f693e29acf2216f1225c2a8f4278aed0f089252138346e28`; pinned code SHA-256: `9ebc90c30036e11934da4e35d209033d6b927409c6cb717d0bf567daf36f3625`.

Changes in this research candidate are limited to the Kaggle owner/slug, title, file name, this explicit provenance notice, an explicit dataset-version pin, and `nbformat_minor` 4→5 so the upstream cell IDs are valid under the notebook schema. No upstream cell ID, source, executable code, output, or execution count was changed. All executable code is byte-for-byte identical to Sakhawat Version 1. Retain this notice and the Apache-2.0 license when redistributing the notebook or a derivative.

The checkpoint is referenced, not redistributed: `tonylica/rsna2026-models`, dataset ID `11546462`, dataset version `2`, Kaggle source/version ID `18706996`, file `rsna_20260807_v1.pt`. Kaggle currently reports that dataset's license as `Unknown`; obtain an explicit license before copying, republishing, or distributing the checkpoint outside its Kaggle dataset. The DINOv2-small dependency is pinned to `metaresearch/dinov2/PyTorch/small/1` (model ID `986`, instance ID `3325`, version source ID `4533`) and is Apache-2.0 licensed.



# RSNA Knee Abnormality Detection — Multi-View DINOv2 Inference

This notebook performs study-level inference from multi-sequence knee MRI series. Each study is reduced to a small set of clinically useful view/sequence slots, representative slices are converted into three-channel inputs, and a DINOv2 encoder produces features that are aggregated by a diagnosis-specific attention head.

### What this version changes

- Keeps the checkpoint-compatible model architecture and preprocessing contract intact.
- Uses DICOM geometry rather than filename order when arranging slices.
- Normalizes right/left knees into a consistent anatomical convention when the metadata supports it.
- Adds **overlapping slice-window test-time averaging**: instead of evaluating only three disjoint 3-slice windows, the model can evaluate every consecutive 3-slice window in the cached 9-slice stack.
- Uses a conservative **hybrid fold ensemble** that is dominated by fold-wise percentile ranks while adding a small rank of the mean-probability signal.
- Keeps the entire inference path offline and Kaggle-compatible.

> **Reproducibility / attribution:** this notebook is an independent cleanup and extension of the supplied baseline implementation. If the original baseline, checkpoint, or model dataset belongs to another Kaggle author, preserve the license and add the original source attribution before publishing. Reformatting or rewriting code does not remove attribution obligations.

The public leaderboard score can move up or down because the hidden evaluation set is different from any local sample. The two inference changes above are intentionally modest and are designed to reduce prediction variance without changing the learned model itself.



## 1. Configuration

The defaults below match the attached model bundle. `OVERLAP_TTA=True` is the main experimental change. For a 9-slice cache and 3-channel inputs it evaluates 7 consecutive windows instead of only 3 disjoint windows.

`FOLD_RANK_WEIGHT=0.90` keeps the final ensemble close to the original rank-averaging behavior. The remaining 10% comes from the rank of the probability-mean ensemble; this can break a few disagreements between folds without making calibration dominate an AUC-oriented submission.


In [ ]:

from __future__ import annotations

import gc
import os
import re
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_var, "4")

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

START_TIME = time.time()
SEED = 20260808
np.random.seed(SEED)
torch.manual_seed(SEED)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

IMG = 224
CROP_MM = 160.0
GROUP = 3
N_GROUP = 3
CACHE_SLICES = GROUP * N_GROUP

HDR_THREADS = 16
PIX_THREADS = 12
EVAL_BATCH = 12
TIME_BUDGET = 8.0 * 3600

BACKBONE_VARIANT = "small"
UNFREEZE_LAST = 6
MODEL_FILE = "rsna_20260807_v1.pt"

# Inference improvements.
OVERLAP_TTA = True
LOGIT_POOL_WEIGHT = 0.90      # 1.0 reproduces sigmoid(mean(logits))
FOLD_RANK_WEIGHT = 0.90       # dominant component of the final AUC-oriented blend
FOLD_SCORE_POWER = 4.0        # used only when fold validation AUC is saved in the bundle

LAT_FALLBACK = "auto"
LAT_MIN_AGREEMENT = 0.85
LAT_MIN_OFFSET_MM = 5.0

SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]

SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]

SLOT_SCHEME = os.environ.get("SLOT_SCHEME", "recovered")
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == "public" else SLOTS_RECOVERED
N_SLOT = len(SLOTS)

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(
    r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
    r"water excit|\btirm\b|\bsting\b|\bfatsup\b"
)
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


def log(message: str) -> None:
    print(f"[{time.time() - START_TIME:7.1f}s] {message}", flush=True)


## 2. Locate competition data and attached weights

In [ ]:

def find_root() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("data"),
        Path("."),
    ]
    for candidate in candidates:
        if (candidate / "test.csv").is_file() and (candidate / "test_series").is_dir():
            return candidate

    base = Path("/kaggle/input")
    if base.is_dir():
        for level1 in sorted(p for p in base.iterdir() if p.is_dir()):
            nested = [level1] + sorted(p for p in level1.iterdir() if p.is_dir())
            for candidate in nested:
                if (candidate / "test.csv").is_file() and (candidate / "test_series").is_dir():
                    return candidate
    raise FileNotFoundError("RSNA knee competition data was not found in the Kaggle input mount.")


def find_dinov2(variant: str = "small") -> Path | None:
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None

    hits: list[Path] = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if "config.json" in files and "dinov2" in root.lower():
            hits.append(Path(root))

    for hit in hits:
        if variant in str(hit).lower():
            return hit
    return hits[0] if hits else None


def find_model_path() -> Path:
    direct = [
        Path("/kaggle/input/datasets/tonylica/rsna2026-models") / MODEL_FILE,
        Path(MODEL_FILE),
    ]
    for candidate in direct:
        if candidate.is_file():
            return candidate

    base = Path("/kaggle/input")
    if base.is_dir():
        for candidate in base.rglob(MODEL_FILE):
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(f"Required model bundle is missing: {MODEL_FILE}")


ROOT = find_root()
log(f"input root: {ROOT}")



## 3. Read DICOM headers and recover sequence semantics

MRI series names are not perfectly standardized across scanners. The header pass therefore combines sequence description, pulse timing, scan options, and the provided anatomical plane table to identify useful fat-suppressed fluid-sensitive and structural series.


In [ ]:

HDR_TAGS = [
    "SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
    "RepetitionTime", "EchoTime", "Laterality", "ImageLaterality",
    "ImagePositionPatient", "PixelSpacing", "Rows", "Columns",
    "RescaleSlope", "RescaleIntercept",
]


def probe_series(item):
    split, study_uid, series_uid, directory = item
    row = {
        "split": split,
        "StudyInstanceUID": study_uid,
        "SeriesInstanceUID": series_uid,
        "dir": directory,
    }
    try:
        files = sorted(e.name for e in os.scandir(directory) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row

        middle = files[len(files) // 2]
        ds = pydicom.dcmread(
            os.path.join(directory, middle),
            stop_before_pixels=True,
            force=True,
        )
        for tag in HDR_TAGS:
            value = getattr(ds, tag, None)
            if value is None:
                row[tag] = None
            elif isinstance(value, (list, tuple)) or type(value).__name__ == "MultiValue":
                row[tag] = "|".join(str(x) for x in value)
            else:
                row[tag] = str(value)
    except Exception as exc:
        row["err"] = str(exc)[:160]
    return row


def scan_series(split: str) -> pd.DataFrame:
    base = ROOT / split
    if not base.is_dir():
        return pd.DataFrame()

    jobs = []
    for study in os.scandir(base):
        if not study.is_dir():
            continue
        for series in os.scandir(study.path):
            if series.is_dir():
                jobs.append((split, study.name, series.name, series.path))

    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe_series, jobs))
    return pd.DataFrame(rows)


def annotate_sequences(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    scan_options = df["ScanOptions"].fillna("").str.upper().str.split("|")
    option_fatsat = scan_options.apply(
        lambda tokens: any(token.strip() in FATSAT_OPTS for token in tokens)
    )
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | option_fatsat

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")

    named_t1 = desc.str.contains(_T1_RX)
    named_t2 = desc.str.contains(_T2_RX)
    named_pd = desc.str.contains(_PD_RX)

    df["weight"] = np.where(
        named_t1 & ~named_t2 & ~named_pd, "T1",
        np.where(
            named_t2 & ~named_pd, "T2",
            np.where(
                named_pd, "PD",
                np.where(
                    gre, "GRE",
                    np.where(tr < 800, "T1", np.where(te > 60, "T2", np.where(tr >= 800, "PD", "UNK"))),
                ),
            ),
        ),
    )
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce",
    )
    return df


## 4. Laterality normalization and slot selection

In [ ]:

def _tag_side(group: pd.DataFrame) -> str | None:
    values = [str(x).strip().upper() for x in group["Laterality"].dropna()]
    if "ImageLaterality" in group.columns:
        values += [str(x).strip().upper() for x in group["ImageLaterality"].dropna()]
    values = [x[0] for x in values if x and x[0] in ("L", "R")]
    return values[0] if values else None


def _position_side(group: pd.DataFrame) -> str | None:
    xs = []
    for raw in group.get("ImagePositionPatient", pd.Series(dtype=object)).dropna():
        try:
            xs.append(float(str(raw).split("|")[0]))
        except Exception:
            pass
    if not xs:
        return None

    median_x = float(np.median(xs))
    if abs(median_x) < LAT_MIN_OFFSET_MM:
        return None
    return "R" if median_x < 0 else "L"  # DICOM patient coordinates use LPS.


def laterality_maps(headers: pd.DataFrame):
    tagged, positioned = {}, {}
    for study_uid, group in headers.groupby("StudyInstanceUID"):
        tagged[study_uid] = _tag_side(group)
        positioned[study_uid] = _position_side(group)

    comparable = [s for s in tagged if tagged[s] and positioned[s]]
    agreement = (
        float(np.mean([tagged[s] == positioned[s] for s in comparable]))
        if comparable else np.nan
    )
    tag_coverage = float(np.mean([v is not None for v in tagged.values()]))

    if LAT_FALLBACK == "on":
        use_position = True
    elif LAT_FALLBACK == "off":
        use_position = False
    else:
        use_position = bool(comparable) and np.isfinite(agreement) and agreement >= LAT_MIN_AGREEMENT

    resolved = {
        study_uid: (tagged[study_uid] or (positioned[study_uid] if use_position else None))
        for study_uid in tagged
    }
    final_coverage = float(np.mean([v is not None for v in resolved.values()]))

    info = {
        "tag_coverage": tag_coverage,
        "agreement": agreement,
        "n_compared": len(comparable),
        "fallback_used": use_position,
        "final_coverage": final_coverage,
    }
    log(
        f"laterality: tag={tag_coverage:.1%}, agreement={agreement:.1%} "
        f"on {len(comparable)} comparable studies, final={final_coverage:.1%}"
    )
    return resolved, info


def pick_slots(series_df: pd.DataFrame, plane_map: dict) -> dict:
    table = series_df.copy()
    table["plane"] = table["SeriesInstanceUID"].map(plane_map)

    selected = {}
    for study_uid, group in table.groupby("StudyInstanceUID"):
        study_slots = {}
        for slot_name, plane, fluid, fatsat in SLOTS:
            keep = (group["plane"] == plane) & (group["fatsat"] == fatsat)
            if fluid is not None:
                keep &= group["fluid"] == fluid

            candidates = group[keep]
            if len(candidates) == 0 and fluid is False:
                candidates = group[(group["plane"] == plane) & (~group["fatsat"])]

            if len(candidates):
                study_slots[slot_name] = candidates.sort_values(
                    "n_slices", ascending=False
                ).iloc[0]
        selected[study_uid] = study_slots
    return selected



## 5. Spatial sorting, physical crop, and study cache

The image pipeline samples the central 60% of each selected MRI stack. A constant physical field of view is used when pixel spacing is available, then every slice is resized to the network input resolution and normalized using per-series robust percentiles.


In [ ]:

def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", str(name)))


def spatially_sorted_files(record) -> list[str]:
    files = list(record["files"])
    directory = record["dir"]
    rows = []

    for original_pos, name in enumerate(files):
        ipp, instance = None, None
        try:
            ds = pydicom.dcmread(
                os.path.join(directory, name),
                stop_before_pixels=True,
                force=True,
                specific_tags=["ImagePositionPatient", "InstanceNumber"],
            )
            raw_ipp = getattr(ds, "ImagePositionPatient", None)
            if raw_ipp is not None and len(raw_ipp) >= 3:
                candidate = np.asarray(raw_ipp[:3], dtype=np.float64)
                if np.isfinite(candidate).all():
                    ipp = candidate
            raw_instance = getattr(ds, "InstanceNumber", None)
            if raw_instance is not None:
                instance = float(raw_instance)
        except Exception:
            pass
        rows.append((name, ipp, instance, original_pos))

    positioned = [row for row in rows if row[1] is not None]
    threshold = max(2, int(0.8 * len(rows)))

    if len(positioned) >= threshold:
        xyz = np.stack([row[1] for row in positioned])
        varying_axis = int(np.argmax(np.ptp(xyz, axis=0)))
        fallback = float(np.nanmedian(xyz[:, varying_axis]))
        rows.sort(
            key=lambda row: (
                float(row[1][varying_axis]) if row[1] is not None else fallback,
                row[2] if row[2] is not None else float("inf"),
                row[3],
            )
        )
    elif sum(row[2] is not None for row in rows) >= threshold:
        rows.sort(key=lambda row: (row[2] if row[2] is not None else float("inf"), row[3]))
    else:
        rows.sort(key=lambda row: _natural_key(row[0]))

    return [row[0] for row in rows]


def read_slot(record, n_slice: int | None = None, out_size: int | None = None):
    n_slice = CACHE_SLICES if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size

    files = spatially_sorted_files(record)
    directory = record["dir"]
    px = record["px"]
    n_files = len(files)
    if n_files == 0:
        return None

    low_idx = int(0.20 * (n_files - 1))
    high_idx = int(0.80 * (n_files - 1))
    if high_idx > low_idx:
        sample_idx = np.unique(np.linspace(low_idx, high_idx, n_slice).astype(int))
    else:
        sample_idx = np.array([n_files // 2])
    while len(sample_idx) < n_slice:
        sample_idx = np.append(sample_idx, sample_idx[-1])

    planes = []
    for index in sample_idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(directory, files[int(index)]), force=True)
            image = ds.pixel_array.astype(np.float32)
            slope = float(getattr(ds, "RescaleSlope", 1) or 1)
            intercept = float(getattr(ds, "RescaleIntercept", 0) or 0)
            image = image * slope + intercept
        except Exception:
            image = np.zeros((out_size, out_size), dtype=np.float32)
        planes.append(image)

    shape = planes[0].shape
    planes = [p if p.shape == shape else np.zeros(shape, np.float32) for p in planes]
    volume = np.stack(planes)

    if px and np.isfinite(px) and px > 0:
        desired = int(round(CROP_MM / px))
        h, w = shape
        if 16 < desired < min(h, w):
            cy, cx = h // 2, w // 2
            half = desired // 2
            volume = volume[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    p01, p99 = np.percentile(volume, [1, 99])
    volume = np.clip((volume - p01) / max(p99 - p01, 1e-6), 0, 1)

    tensor = torch.from_numpy(np.ascontiguousarray(volume)).unsqueeze(0)
    tensor = F.interpolate(
        tensor, size=(out_size, out_size), mode="bilinear", align_corners=False
    )
    return (tensor.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


def normalise_laterality(image: torch.Tensor, plane: str, laterality: str | None):
    if laterality != "R":
        return image
    if plane in ("Coronal", "Axial"):
        return torch.flip(image, dims=[-1])
    return torch.flip(image, dims=[0])


def build_cache(slot_map: dict, laterality_map: dict, tag: str):
    studies = sorted(slot_map)
    study_index = {study_uid: i for i, study_uid in enumerate(studies)}

    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), dtype=np.uint8)
    mask = np.zeros((len(studies), N_SLOT), dtype=np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024**3:.2f} GB")

    jobs = [
        (study_uid, slot_idx, plane, slot_map[study_uid][slot_name])
        for study_uid in studies
        for slot_idx, (slot_name, plane, _, _) in enumerate(SLOTS)
        if slot_name in slot_map[study_uid]
    ]
    log(f"{tag}: decoding {len(jobs)} selected series")

    chunk_size = 512
    completed = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for start in range(0, len(jobs), chunk_size):
            block = jobs[start:start + chunk_size]
            decoded = pool.map(lambda job: read_slot(job[3], CACHE_SLICES, IMG), block)

            for (study_uid, slot_idx, plane, _), image in zip(block, decoded):
                completed += 1
                if image is None:
                    continue
                image = normalise_laterality(image, plane, laterality_map.get(study_uid))
                cache[study_index[study_uid], slot_idx] = image.numpy()
                mask[study_index[study_uid], slot_idx] = 1.0

            if completed % 4096 < chunk_size:
                log(f"  {tag}: {completed}/{len(jobs)} series decoded")
            if time.time() - START_TIME > TIME_BUDGET:
                log(f"  {tag}: time budget reached during DICOM decoding")
                break

    gc.collect()
    return studies, cache, mask



## 6. Checkpoint-compatible model

The module names and tensor shapes in this section intentionally match the saved checkpoint. The head learns a separate attention distribution over MRI slots for every diagnosis, while the DINOv2 representation combines the CLS token, average patch representation, and a focal top-k patch summary.


In [ ]:

class SlotHead(nn.Module):
    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

        prior = torch.zeros(n_out, n_slot)
        if SLOT_SCHEME == "recovered" and n_slot == 6 and n_out == len(TARGETS):
            preferred = {
                "ACL": (0, 3, 5),
                "MCL": (1, 4),
                "Medial Meniscus": (0, 1, 3, 4),
                "Lateral Meniscus": (0, 1, 3, 4),
                "Medial OA": (1, 4, 5),
                "Lateral OA": (1, 4, 5),
                "PF OA": (0, 2, 5),
                "Effusion": (0, 2),
                "Synovitis": (0, 2),
                "Baker's": (0,),
                "Contusion": (0, 1, 2),
                "Fracture": (0, 1, 2, 4, 5),
            }
            for target, slots in preferred.items():
                prior[TARGETS.index(target), list(slots)] = 0.55
        self.register_buffer("slot_prior", prior)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        attention = (
            torch.einsum("bsh,oh->bos", h, self.query) / self.hidden**0.5
            + self.slot_prior.unsqueeze(0)
        )
        attention = attention.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        context = self.drop(torch.einsum("bos,bsh->boh", attention, h))
        return (context * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class Model(nn.Module):
    def __init__(self, backbone, dim):
        super().__init__()
        self.backbone = backbone
        self.head = SlotHead(dim, N_SLOT, len(TARGETS))
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask):
        batch, slots = imgs.shape[:2]
        x = imgs.reshape(batch * slots, *imgs.shape[2:]).float().div_(255.0)
        x = (x - self.mean) / self.std

        encoded = self.backbone(pixel_values=x).last_hidden_state
        patches = encoded[:, 1:]
        k = max(1, patches.shape[1] // 8)
        focal = patches.topk(k, dim=1).values.mean(1)
        features = torch.cat([encoded[:, 0], patches.mean(1), focal], dim=1)
        features = features.reshape(batch, slots, -1)
        return self.head(features, mask)


def build_model():
    from transformers import AutoModel

    backbone_path = find_dinov2(BACKBONE_VARIANT)
    if backbone_path is None:
        raise FileNotFoundError("Offline DINOv2 weights are not attached to this Kaggle notebook.")

    backbone = AutoModel.from_pretrained(str(backbone_path))
    n_layers = len(backbone.encoder.layer)

    for parameter in backbone.parameters():
        parameter.requires_grad = False
    for block in backbone.encoder.layer[max(0, n_layers - UNFREEZE_LAST):]:
        for parameter in block.parameters():
            parameter.requires_grad = True
    for parameter in backbone.layernorm.parameters():
        parameter.requires_grad = True

    feature_dim = backbone.config.hidden_size * 3
    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    log(
        f"backbone: {n_layers} blocks, last {UNFREEZE_LAST} trainable "
        f"({trainable / 1e6:.1f}M params), feature dim={feature_dim}"
    )
    return Model(backbone, feature_dim)



## 7. Enhanced test-time aggregation

The baseline cache contains 9 slices per slot and feeds them to the model as three non-overlapping triplets. Here, the default inference evaluates all 7 consecutive triplets: `[0:3]`, `[1:4]`, …, `[6:9]`. This reuses the same decoded cache, so there is no additional DICOM I/O.

For each fold, most of the prediction comes from `sigmoid(mean(logits))`, matching the original behavior. A small mean-probability component is included to reduce sensitivity to one extreme window.


In [ ]:

def require_cuda() -> torch.device:
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required for this inference notebook.")

    device_name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    arch = f"sm_{capability[0]}{capability[1]}"
    supported = set(torch.cuda.get_arch_list())
    log(f"cuda: {device_name}, capability={arch}")
    if arch not in supported:
        raise RuntimeError(f"The installed PyTorch build does not support the assigned GPU ({arch}).")

    torch.backends.cuda.matmul.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
    return torch.device("cuda")


def load_bundle():
    path = find_model_path()
    log(f"model bundle: {path}")
    try:
        bundle = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        bundle = torch.load(path, map_location="cpu")
    return bundle


def apply_bundle_config(bundle):
    global TARGETS, SLOTS, N_SLOT, IMG, GROUP, N_GROUP, CACHE_SLICES, BACKBONE_VARIANT

    TARGETS = list(bundle.get("targets", TARGETS))
    SLOTS = [tuple(slot) for slot in bundle.get("slots", SLOTS)]
    N_SLOT = len(SLOTS)
    IMG = int(bundle.get("img", IMG))
    GROUP = int(bundle.get("group", GROUP))
    N_GROUP = int(bundle.get("n_group", N_GROUP))
    CACHE_SLICES = GROUP * N_GROUP

    variant = str(bundle.get("model_variant", "dinov2-small")).split("-")[-1]
    BACKBONE_VARIANT = "base" if variant == "base" else "small"
    log(
        f"bundle config: backbone={BACKBONE_VARIANT}, img={IMG}, "
        f"group={GROUP}, cached_slices={CACHE_SLICES}, slots={N_SLOT}"
    )


def window_starts() -> list[int]:
    if OVERLAP_TTA and CACHE_SLICES >= GROUP:
        return list(range(CACHE_SLICES - GROUP + 1))
    return [group_idx * GROUP for group_idx in range(N_GROUP)]


def take_window(cache_rows: torch.Tensor, start: int) -> torch.Tensor:
    return cache_rows[:, :, start:start + GROUP]


@torch.no_grad()
def predict_fold(model, cache, mask, indices, device):
    model.eval()
    starts = window_starts()
    outputs = []

    for batch_start in range(0, len(indices), EVAL_BATCH):
        batch_idx = indices[batch_start:batch_start + EVAL_BATCH]
        rows = torch.from_numpy(cache[batch_idx]).to(device, non_blocking=True)
        slot_mask = torch.from_numpy(mask[batch_idx]).to(device, non_blocking=True)

        logit_sum = None
        probability_sum = None
        for start in starts:
            with torch.autocast("cuda", enabled=device.type == "cuda"):
                logits = model(take_window(rows, start), slot_mask).float()
            probabilities = torch.sigmoid(logits)
            logit_sum = logits if logit_sum is None else logit_sum + logits
            probability_sum = probabilities if probability_sum is None else probability_sum + probabilities

        mean_logits = logit_sum / len(starts)
        mean_probabilities = probability_sum / len(starts)
        pooled = (
            LOGIT_POOL_WEIGHT * torch.sigmoid(mean_logits)
            + (1.0 - LOGIT_POOL_WEIGHT) * mean_probabilities
        )
        outputs.append(pooled.cpu().numpy())

    if not outputs:
        return np.zeros((0, len(TARGETS)), dtype=np.float32)
    return np.concatenate(outputs, axis=0)


def percentile_rank(matrix: np.ndarray) -> np.ndarray:
    return pd.DataFrame(matrix).rank(axis=0, pct=True, method="average").to_numpy(dtype=np.float64)


def extract_fold_quality(fold: dict) -> float | None:
    # Only accept explicit AUC-like metadata; generic loss values are intentionally ignored.
    for key in ("val_auc", "macro_auc", "auc", "best_auc", "valid_auc"):
        value = fold.get(key)
        if isinstance(value, (int, float, np.number)) and np.isfinite(value) and 0.5 <= float(value) <= 1.0:
            return float(value)
    return None


def fold_weights(fold_states: list[dict]) -> np.ndarray:
    quality = [extract_fold_quality(fold) for fold in fold_states]
    if any(value is None for value in quality):
        log("fold validation AUC metadata not available -> equal fold weights")
        return np.ones(len(fold_states), dtype=np.float64)

    raw = np.asarray(quality, dtype=np.float64) ** FOLD_SCORE_POWER
    raw /= raw.mean()
    log("fold weights from saved validation AUC: " + ", ".join(f"{w:.3f}" for w in raw))
    return raw


## 8. Run inference and write `submission.csv`

In [ ]:

def write_submission(final_scores: np.ndarray, study_uids: list[str], test_df: pd.DataFrame):
    prediction_table = pd.DataFrame(final_scores, columns=TARGETS)
    prediction_table.insert(0, "StudyInstanceUID", study_uids)
    prediction_table["StudyInstanceUID"] = prediction_table["StudyInstanceUID"].astype(str)

    submission = test_df[["StudyInstanceUID"]].merge(
        prediction_table, on="StudyInstanceUID", how="left"
    )
    submission[TARGETS] = submission[TARGETS].fillna(0.5)
    submission.to_csv("submission.csv", index=False)
    return submission


def main():
    device = require_cuda()
    bundle = load_bundle()
    apply_bundle_config(bundle)

    test_df = pd.read_csv(ROOT / "test.csv")
    test_df["StudyInstanceUID"] = test_df["StudyInstanceUID"].astype(str)

    test_series = pd.read_csv(ROOT / "test_series.csv")
    test_series["StudyInstanceUID"] = test_series["StudyInstanceUID"].astype(str)
    test_series["SeriesInstanceUID"] = test_series["SeriesInstanceUID"].astype(str)
    log(f"test={test_df.shape}; test_series={test_series.shape}")

    plane_map = dict(zip(test_series["SeriesInstanceUID"], test_series["Anatomical_Plane"]))

    log("reading test DICOM headers")
    headers = annotate_sequences(scan_series("test_series"))
    log(f"header rows: {len(headers)}")

    laterality, lat_info = laterality_maps(headers)
    log(f"laterality diagnostics: {lat_info}")

    slot_map = pick_slots(headers, plane_map)
    slot_counts = pd.Series([len(slots) for slots in slot_map.values()])
    if len(slot_counts):
        log(
            f"slots/study: mean={slot_counts.mean():.2f}, "
            f"min={slot_counts.min():.0f}, max={slot_counts.max():.0f}"
        )

    study_uids, cache, mask = build_cache(slot_map, laterality, "test")

    fold_states = bundle.get("fold_states", [])
    if not fold_states:
        raise ValueError("The saved model bundle contains no fold_states.")

    weights = fold_weights(fold_states)
    total_weight = float(weights.sum())
    rank_accumulator = np.zeros((len(study_uids), len(TARGETS)), dtype=np.float64)
    probability_accumulator = np.zeros_like(rank_accumulator)

    log(f"TTA windows per fold: {len(window_starts())} -> {window_starts()}")

    all_indices = np.arange(len(study_uids))
    for fold_number, (fold, weight) in enumerate(zip(fold_states, weights), start=1):
        model = build_model().to(device)
        model.load_state_dict(fold["state_dict"], strict=True)

        probabilities = predict_fold(model, cache, mask, all_indices, device)
        rank_accumulator += weight * percentile_rank(probabilities)
        probability_accumulator += weight * probabilities

        fold_id = fold.get("fold", fold_number - 1)
        log(f"inferred fold {fold_id} ({fold_number}/{len(fold_states)}), weight={weight:.3f}")

        del model
        gc.collect()
        torch.cuda.empty_cache()

    mean_fold_rank = rank_accumulator / total_weight
    mean_probability = probability_accumulator / total_weight
    probability_rank = percentile_rank(mean_probability)

    final_scores = (
        FOLD_RANK_WEIGHT * mean_fold_rank
        + (1.0 - FOLD_RANK_WEIGHT) * probability_rank
    )

    submission = write_submission(final_scores, study_uids, test_df)
    log(f"submission.csv written: {submission.shape}")
    print(submission.head().to_string(index=False))
    return submission


In [ ]:

try:
    submission = main()
except Exception:
    traceback.print_exc()
    fallback = pd.read_csv(find_root() / "test.csv")
    for target in TARGETS:
        fallback[target] = 0.5
    fallback.to_csv("submission_fallback.csv", index=False)
    print("A fallback file was written for debugging; the notebook is re-raising the error.")
    raise

log("done")
